We have extracted data for the test dataset now (step 1-2) and saved it as JSON files in 
'/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/' 
and '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C and D'

This notebook addresses step 3 (and probably also step 5). 

3. Per program, extract the following info for courses included in the program (+ suggested electives?)
   Also do this for the test dataset.
   { 

  "title": "...", 

  "credits": "...", 

  "level": "BSc / MSc / ...", 

“description: “,  

  "topics": ["...", "..."],           // extracted or inferred (LLM / SWEBOK?) 

  "learning_outcomes": ["...", "..."], // explicit or inferred from exam/content description 

  "prerequisites": ["...", "..."], 

  "examination_methods": ["...", "..."], 

}  

4. Find similarity between courses in a hierarchical way (start with description, topics and/or LOs. Then look at prerequisites of one course vs LOs/descriptions of another course, level and examination methods. 

5. Do this for the test dataset and see whether the similarity matches what we would expect. Analyze this per data source (LO, description, topic). Is there a difference between:

- embedding model (sentenceBERT vs TF-IDF vs bigger document-level embedding) 
- data source (only LO matching, description, topic)
- topic extraction method (LLM / SWEBOK)
Other idea for test case is Dutch TUs. 

In [ ]:
# load some sort of open-source LLM or DeepSeek 
# extract a unified format
# extract / including topics from SWEBOK / open topics by LLM 
# visualize 'cloud' of topics and how it differs between C and D program? 
# visualize Delft courses together with LU courses based on these topics 
# match Delft courses to closest alternative(s) in LU based on title/ learning objectives / topics / description / a combination
# we need to do this for all elective courses! + add ovrigt 

In [1]:
#!pip install sentence-transformers==2.2.2

In [17]:
# DeepSeek
from openai import OpenAI
# for DeepSeek-V3, set model='deepseek-chat' 
# for DeepSeek-R1, set model='deepseek-reasoner' 
def extract_course_info_from_json_prompt(json_file):
    """
    Extracts course information from a parsed JSON file containing webpage content.
    Returns a dictionary with title, credits, level, description, topics,
    learning_outcomes, prerequisites, and examination_methods.
    """
    
    base_instructions = (
        "You are an expert curriculum analyst and course information extractor with a specialization in computer science education.\n"
        "Instructions:\n"
        "a. Carefully read the provided parsed JSON content (extracted from a course webpage).\n"
        "b. Extract the following fields if available:\n"
        "   - title: course title\n"
        "   - credits: credit value (e.g., '5 ECTS', '3 credits')\n"
        "   - level: education level (e.g., 'BSc', 'MSc', 'PhD', 'Advanced', 'Intermediate')\n"
        "   - description: course description text\n"
        "   - topics: list of main topics covered (infer from syllabus, content list, or description)\n"
        "   - learning_outcomes: list of learning outcomes (infer from 'learning objectives', 'students will be able to', or course goals)\n"
        "   - prerequisites: list of required prior knowledge/courses (may be labeled as 'prerequisites', 'required knowledge', 'entry requirements', 'prior knowledge')\n"
        "   - examination_methods: list of assessment methods (may be labeled as 'assessment', 'examination', 'evaluation', 'grading')\n"
        "c. If a field is not explicitly present or cannot be inferred, leave it as empty string (for single fields) or empty list (for list fields).\n"
        "d. Be flexible with section headers — the same information may appear under different names.\n"
        "e. Infer learning outcomes from phrases like 'After this course, students will be able to...', 'Upon completion...', or from exam tasks.\n"
        "f. Infer topics from table of contents, syllabus sections, or repeated concepts in the description.\n"
        "g. Do NOT invent information that is not present or reasonably inferable.\n"
        "h. Return ONLY a valid dictionary in the exact format shown below.\n"
        "i. Do NOT include any explanation, commentary, or additional text outside the dictionary.\n"
    )
    
    output_format = {
        "title": "",
        "credits": "",
        "level": "",
        "description": "",
        "topics": [],
        "learning_outcomes": [],
        "prerequisites": [],
        "examination_methods": []
    }
    
    extraction_prompt = [
        {"role": "system", "content": base_instructions},
        {"role": "user", "content": 
         f"# Webpage content (JSON format):\n{json_file}\n\n"
         f"# Extract course information and return as JSON matching this structure:\n"
         f"{output_format}\n\n"
         f"# Rules:\n"
         f"- Use empty string '' for missing single-value fields\n"
         f"- Use empty list [] for missing multi-value fields\n"
         f"- For 'level', standardize to 'BSc', 'MSc', 'PhD', 'Bachelor', 'Master', etc.\n"
         f"- For 'topics', extract as list of strings\n"
         f"- For 'learning_outcomes', extract as list of strings\n"
         f"- For 'prerequisites', extract as list of strings\n"
         f"- For 'examination_methods', extract as list of strings (e.g., 'written exam', 'oral presentation', 'project', 'homework')\n"
         f"- Return ONLY the JSON, no other text."}
    ]
    
    return extraction_prompt 

### Try scraping LU webpages 
import playwright
from playwright.async_api import async_playwright
import pandas as pd

async def scrape_lth_courses_direct(url):
    """
    Scrape courses by waiting for the specific table structure
    """
    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        page = await browser.new_page()
        
        # Navigate with specific wait conditions
        await page.goto(url)
        
        # Wait for any content to load
        await page.wait_for_timeout(5000)
        
        # Get all text content
        content = await page.content()
        
        # Look for course codes in the page source
        course_codes = await page.evaluate('''
            () => {
                const text = document.body.innerText;
                const codePattern = /[A-Z]{2,5}\\d{2,5}/g;
                const matches = text.match(codePattern) || [];
                return [...new Set(matches)];  // Remove duplicates
            }
        ''')
        
        # Build course URLs from codes
        courses = []
        for code in course_codes:
            if len(code) >= 5 and len(code) <= 8:  # Filter valid course codes
                courses.append({
                    'course_code': code,
                    'url': f'https://kurser.lth.se/lot/course/{code}',
                    'url_kp': f'https://kurser.lth.se/lot/course-syllabus-en/26_27/{code}',
                    'programme': 'C',
                    'academic_year': '26_27'
                })
        
        await browser.close()
        
        # Create DataFrame
        df = pd.DataFrame(courses)
        return df
        
url = 'https://kurser.lth.se/lot/programme?programme=C&ay=26_27#EITA55'
df = await scrape_lth_courses_direct(url)
print(df)

    course_code                                      url  \
0        FMAB65  https://kurser.lth.se/lot/course/FMAB65   
1        EDAB05  https://kurser.lth.se/lot/course/EDAB05   
2        FMAB70  https://kurser.lth.se/lot/course/FMAB70   
3        EITA66  https://kurser.lth.se/lot/course/EITA66   
4        EDAA01  https://kurser.lth.se/lot/course/EDAA01   
..          ...                                      ...   
170      EITL01  https://kurser.lth.se/lot/course/EITL01   
171      MAML15  https://kurser.lth.se/lot/course/MAML15   
172      EXTL03  https://kurser.lth.se/lot/course/EXTL03   
173      FMAL01  https://kurser.lth.se/lot/course/FMAL01   
174      FMSL01  https://kurser.lth.se/lot/course/FMSL01   

                                                url_kp programme academic_year  
0    https://kurser.lth.se/lot/course-syllabus-en/2...         C         26_27  
1    https://kurser.lth.se/lot/course-syllabus-en/2...         C         26_27  
2    https://kurser.lth.se/lot/cours

In [13]:
#query = extract_course_info_from_json_prompt(json_file)
#print(query)
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/'

def find_json_files(directory_path, recursive=True):
    """Find all JSON files in the specified directory."""
    json_files = []
    
    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist")
        return json_files
    
    try:
        if recursive:
            for root, dirs, files in os.walk(directory_path):
                for file in files:
                    if file.lower().endswith('.json'):
                        full_path = os.path.join(root, file)
                        json_files.append(full_path)
        else:
            for item in os.listdir(directory_path):
                item_path = os.path.join(directory_path, item)
                if os.path.isfile(item_path) and item.lower().endswith('.json'):
                    json_files.append(item_path)
    except Exception as e:
        print(f"Error searching for JSON files: {e}")
    
    return json_files

def load_json_file(file_path):
    """Load and parse a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data
    except json.JSONDecodeError as e:
        print(f"JSON decode error in {file_path}: {e}")
        return None
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

json_files = find_json_files(output_path)
print('first json file: ',json_files[0])
parsed_json = load_json_file(json_files[0])
#print(parsed_json)
prompt = extract_course_info_from_json_prompt(parsed_json)
#print(prompt)

first json file:  /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MAMF60.json
[{'role': 'system', 'content': "You are an expert curriculum analyst and course information extractor with a specialization in computer science education.\nInstructions:\na. Carefully read the provided parsed JSON content (extracted from a course webpage).\nb. Extract the following fields if available:\n   - title: course title\n   - credits: credit value (e.g., '5 ECTS', '3 credits')\n   - level: education level (e.g., 'BSc', 'MSc', 'PhD', 'Advanced', 'Intermediate')\n   - description: course description text\n   - topics: list of main topics covered (infer from syllabus, content list, or description)\n   - learning_outcomes: list of learning outcomes (infer from 'learning objectives', 'students will be able to', or course goals)\n   - prerequisites: list of required prior knowledge/courses (may be labeled as 'prerequisites', 'required knowledge', 'entry requirements', 'prior knowledge')\n

In [31]:
# find all mandatory courses for the C program
# specialization course 
# and elective course 
import ast 

mandatory_courses = df['course_code'] #.iloc[:30]
#print(mandatory_courses)
all_courses = [] 
failed_courses = [] 
for course in mandatory_courses: 
    try: 
        json_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/' + course + '.json' # find json filepath
        print(json_path)
        parsed_json = load_json_file(json_path) # parse json file 
        prompt = extract_course_info_from_json_prompt(parsed_json) # create prompt 
        response = client.chat.completions.create(
        model="deepseek-chat",
        messages = prompt, 
        stream = False) # pass prompt to DeepSeek 
        #print(response.choices[0].message.content) # print the output
        response_text = response.choices[0].message.content
        course_dict = ast.literal_eval(response_text) 
        print(course_dict['title'])
        course_dict['course_code'] = course 
        all_courses.append(course_dict) 
    except SyntaxError as e:
        print(f"✗ Syntax error for {course}: {e}")
        failed_courses.append(course)
        # Print the problematic response for debugging
        print(f"Problematic response: {response_text}")
        
    except Exception as e:
        print(f"✗ Unexpected error for {course}: {e}")
        failed_courses.append(course)

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMAB65.json
Calculus in One Variable B1
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAB05.json
Introduction to Programming
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMAB70.json
Calculus in One Variable B2
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EITA66.json
Design of Systems for Digital Transformation
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAA01.json
Programming - Second Course
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MAMA20.json
Cognition and Interaction Design
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAA75.json
Discrete Structures
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EITA55.json
Communication Systems
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAF60.json
Object-oriented Modelling and Design
/mimer/NOBACKUP/grou

In [32]:
df_all_courses = pd.DataFrame(all_courses)

# Save to files
df_all_courses.to_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/all_mandatory_courses.csv', index=False, encoding='utf-8')
df_all_courses.to_pickle('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/all_mandatory_courses.pkl')

print(f"\n✓ Saved {len(all_courses)} courses successfully")
print(f"✗ Failed: {len(failed_courses)} courses")
print(f"\nDataFrame shape: {df_all_courses.shape}")
print(f"Columns: {df_all_courses.columns.tolist()}")
print("\nFirst row preview:")
print(df_all_courses.iloc[0])


✓ Saved 175 courses successfully
✗ Failed: 0 courses

DataFrame shape: (175, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
title                                        Calculus in One Variable B1
credits                                                      7.5 credits
level                                                                BSc
description            The aim of the course is to give a basic intro...
topics                 [Number concept, Calculation with fractions, I...
learning_outcomes      [within the framework of the course with confi...
prerequisites                                                         []
examination_methods    [Written test comprising theory and problem so...
course_code                                                       FMAB65
Name: 0, dtype: object


In [39]:
# now also take all test courses 
# Define your multiple directories
directories_to_search = [
    '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/',
    '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/UCSC/student1/',
    '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/UCSC/student2/',
    '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUM/student1/',
    '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUM/student2/'
    # Add more directories as needed
]

# Find all JSON files across all directories
all_json_files = []
for directory in directories_to_search:
    json_files = find_json_files(directory, recursive=True)
    all_json_files.extend(json_files)

print(f"Found {len(all_json_files)} JSON files across {len(directories_to_search)} directories\n")

# Extract course names from each file
all_courses = [] 
failed_courses = [] 
for file_path in all_json_files:
    try: 
        # Get just the filename without path
        filename = os.path.basename(file_path)
        print(file_path)
        # Extract course name (everything before .json)
        course_name = os.path.splitext(os.path.basename(file_path))[0]
        
        parsed_json = load_json_file(file_path) # parse json file 
        prompt = extract_course_info_from_json_prompt(parsed_json) # create prompt 
        response = client.chat.completions.create(
        model="deepseek-chat",
        messages = prompt, 
        stream = False) # pass prompt to DeepSeek 
        #print(response.choices[0].message.content) # print the output
        response_text = response.choices[0].message.content
        course_dict = ast.literal_eval(response_text) 
        print(course_dict['title'])
        course_dict['course_code'] = course_name
        all_courses.append(course_dict) 
    except SyntaxError as e:
        print(f"✗ Syntax error for {course}: {e}")
        failed_courses.append(course)
        # Print the problematic response for debugging
        print(f"Problematic response: {response_text}")
        
    except Exception as e:
        print(f"✗ Unexpected error for {course}: {e}")
        failed_courses.append(course)


Found 29 JSON files across 5 directories

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Systems Security.json
Systems Security
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Smart Phone Sensing.json
Smart Phone Sensing
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Research in Cyber Security – Hacking Lab.json
Research in Cyber Security – Hacking Lab
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Error Correcting Codes.json
Error Correcting Codes
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Distributed Data Systems.json
Distributed Data Systems
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/UCSC/student1/University of California Santa Cruz - CSE160.json
Introduction to Computer Graphics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_N

In [41]:
df_all_courses = pd.DataFrame(all_courses)

# Save to files
df_all_courses.to_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/test_courses.csv', index=False, encoding='utf-8')
df_all_courses.to_pickle('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/test_courses.pkl')

print(f"\n✓ Saved {len(all_courses)} courses successfully")
print(f"✗ Failed: {len(failed_courses)} courses")
print(f"\nDataFrame shape: {df_all_courses.shape}")
print(f"Columns: {df_all_courses.columns.tolist()}")
print("\nFirst row preview:")
print(df_all_courses.iloc[0])


✓ Saved 29 courses successfully
✗ Failed: 0 courses

DataFrame shape: (29, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
title                                                   Systems Security
credits                                                           5 ECTS
level                                                             Master
description            The course overviews principles, concepts, met...
topics                 [Hardware primitives, Operating systems abstra...
learning_outcomes      [Study security architecture of computer syste...
prerequisites          [Operating Systems (CSE2430), Computer Organiz...
examination_methods    [Written Exam (weighting 50%), Group Assignmen...
course_code                  Studiegids — Study Guide _ Systems Security
Name: 0, dtype: object


In [15]:
# DeepSeek
from openai import OpenAI
# for DeepSeek-V3, set model='deepseek-chat' 
# for DeepSeek-R1, set model='deepseek-reasoner' 
response = client.chat.completions.create(
    model="deepseek-chat",
    messages = prompt, 
    stream = False)
print(response.choices[0].message.content)

{'title': 'Interaction Design, Process Course', 'credits': '7.5 credits', 'level': 'First Cycle', 'description': 'The student shall acquire an overview of the multi-disciplinary area interaction design with a focus on the usability-oriented design process. The student will, through a mix of theory and practice, acquire skills to design usable interactive products and services.', 'topics': ['Usability', 'user experience', 'the design process', 'user studies', 'usability goals', 'conceptual design', 'interaction techniques', 'usability evaluation'], 'learning_outcomes': ['explain the principles behind a usability-oriented design process', 'differentiate phases of the design process', 'describe and discuss established methods in the field of interaction design', 'account for similarities and differences in different usability-oriented design methods', "identify users' needs and requirements", "plan design projects with regard to designing usability requirements and meeting users' needs", 